In [ ]:
# Optional configuration for the classic Notebook / RISE extension.
try:
    from notebook.services.config import ConfigManager
except ImportError:
    pass  # Modern Jupyter and HTML slide export do not need this setup.
else:
    ConfigManager().update('livereveal', {
        'theme': 'white', 'transition': 'none',
        'controls': 'false', 'progress': 'true',
    })

In [2]:
%%html
<script>
  function code_toggle() {
    if (code_shown){
      $('div.input').hide('500');
      $('#toggleButton').val('Show Code')
    } else {
      $('div.input').show('500');
      $('#toggleButton').val('Hide Code')
    }
    code_shown = !code_shown
  }

  $( document ).ready(function(){
    code_shown=false;
    $('div.input').hide()
  });
</script>
<form action="javascript:code_toggle()"><input type="submit" id="toggleButton" value="Show Code"></form>

In [ ]:
from IPython.display import SVG, display

# Word and Sentence Embeddings


## Learning objectives

- Read the two embedding tables in word2vec and trace a training update.
- Explain how count-based and prediction-based methods use context.
- Compare vectors using cosine similarity and identify its limits.
- Build a sentence embedding by **mean pooling of static word embeddings**: averaging the word vectors.

## Why representations matter

A model needs numbers. Which similarities should those numbers express?

| Representation | What it preserves |
| :--- | :--- |
| One-hot word vector | Word identity |
| Context-count row | Which words occur nearby |
| Dense learned word vector | Patterns compressed into a small space |
| Mean of word vectors | A sentence's words, without their order |

## One-hot word representations

Give each vocabulary item its own dimension:

$$f(w) \in \{0,1\}^{|V|}$$

One-hot vectors preserve word identity, but every pair of different words is equally dissimilar and the vectors grow with the vocabulary.


## Cosine similarity

For vectors $u$ and $v$:

$$\cos(u,v)=\frac{u\cdot v}{\lVert u\rVert\,\lVert v\rVert}$$

Cosine similarity compares direction rather than magnitude. It is useful only when the geometry of the representation has been learned for a relevant objective.


## Cosine showdown: vote A, B, C or D

<img src="../img/word2vec_2027/cosine_quiz.svg" alt="Query q=(1,1); candidates A=(3,3), B=(1,0.4), C=(5,-1), D=(-1,-1). Arrows start at the origin." width="1100">

**Which candidate is most similar to $q$ by cosine?** Discuss with a neighbour: 20 seconds.

**A wins:** $\cos(q,A)=1$. The other scores are B: $0.919$, C: $0.555$, D: $-1$.

**Twist:** replace B with $10B$. Does the winner change?

**No.** Positive scaling preserves direction. In the original plot, B is closest by Euclidean distance and C is longest. Neither beats A by cosine.

Ask for a simultaneous A/B/C/D vote before revealing. All four options are nonzero vectors. Hint if needed: cosine compares the angle from the origin, not the distance between arrowheads. A lies on the same ray as q. B is a plausible distractor because it is closest in Euclidean distance (0.6). C is longest (sqrt(26)) but points away from q. D points in the opposite direction. Reveal A and the scores, then take a second vote on scaling B before revealing the final sentence. The figure generator computes the scores from these coordinates.

## Dense static word embeddings

Store one learned $d$-dimensional vector for each vocabulary item:

$$E \in \mathbb{R}^{|V|\times d}, \qquad f(w)=E_{w,:}$$

**Static** means every occurrence of a word uses the same row. The dimensions are learned features, not vocabulary items.

## Dense word embeddings in two dimensions

<center><img src="../img/dense_continuous.svg" width="80%"></center>

This is an illustrative two-dimensional space, not the output of the toy training run below. Dense describes the representation; it does not specify how it was learned. Count-based methods can also produce dense vectors.

## The distributional hypothesis

Words that occur in similar contexts often have related meanings.

> You shall know a word by the company it keeps.

Firth (1957)

This is an empirical shortcut, not a complete theory of meaning: corpus choice and social patterns shape the resulting space.


## One corpus, many context pairs

<img src="../img/word2vec_2027/corpus_pairs.svg" alt="Toy corpus: apples are tasty four times, oranges are tasty four times, rabbits are furry twice, hamsters are furry twice. A radius-two window around apples produces (apples, are) and (apples, tasty)." width="1100">

Use a **symmetric window of radius 2**, within each sentence. Every word takes a turn as the centre.

A centre word is also called a target word. Here we say centre and context to keep the two roles clear. Twelve three-token sentences give 36 tokens and 72 ordered centre–context pairs. The window never crosses a sentence boundary. For example, the first sentence contributes (apples, are), (apples, tasty), (are, apples), (are, tasty), (tasty, apples), (tasty, are). Repeated sentences contribute repeated pairs. The code generates pairs directly from tokens and only then accumulates a count matrix for inspection.

## Count-based word vectors: read a row

<img src="../img/word2vec_2027/counts.svg" alt="Seven by seven word-context count matrix. Apples and oranges share the row with counts four for are and tasty. Rabbits and hamsters share the row with counts two for are and furry." width="1100">

$C_{w,c}$ counts observed pairs. Similar rows mean similar context distributions.

Rows and columns use the same seven-word vocabulary but have different roles. C is symmetric here because the window is symmetric. This need not hold for position-sensitive or directional contexts. Apples and oranges never directly co-occur, but their context rows match: distributional similarity is about shared contexts, not just direct co-occurrence.

## Count-based does not have to mean sparse

<img src="../img/word2vec_2027/count_to_dense.svg" alt="Context counts become positive pointwise mutual information, then truncated singular value decomposition gives low-dimensional dense word vectors." width="1100">

**Positive pointwise mutual information (PPMI)** weights association; **singular value decomposition (SVD)** compresses the matrix.

For total pair count N, row count C_w and column count C_c, PMI(w,c) = log(C_wc N / (C_w C_c)); PPMI clips negative PMI to zero and assigns zero to unobserved pairs. Here N=72, C_apples=8, C_are=24 and C_tasty=16. Both pairs have count 4, but PMI(apples,are)=log(1.5)=0.405 and PMI(apples,tasty)=log(2.25)=0.811. The less frequent context is more informative. The diagram shows a real rank-two truncated SVD of this corpus's PPMI matrix, using U_2 Sigma_2 as the word vectors. Other scalings of the singular values are possible; the axes have no intrinsic names. This is still a count-based method even though its final vectors are dense. Source: [Levy & Goldberg (2014)](https://papers.nips.cc/paper_files/paper/2014/file/b78666971ceae55a8e87efb7cbfd9ad4-Paper.pdf).

## Word contexts and document counts are different

| Matrix | A row represents | A column represents |
| :--- | :--- | :--- |
| Word–context counts | A word | A nearby word |
| Document–term counts | A document | A word in the vocabulary |

**Term frequency–inverse document frequency (TF–IDF)** downweights words found in many documents. Lab 3 uses it with a linear classifier.

For scikit-learn's default smoothed IDF, idf(w)=log((1+N)/(1+df(w)))+1; multiply by term counts and apply the configured row normalization (L2 by default). Document frequency counts documents containing w, not the number of occurrences. This document representation is distinct from the word-context rows used in this lecture.

## Word2vec: two prediction architectures

<img src="../img/word2vec_2027/cbow_skipgram.svg" alt="Continuous bag-of-words averages context input vectors and predicts the centre. Skip-gram looks up the centre input vector and predicts each surrounding context." width="1100">

**Continuous bag-of-words (CBOW):** context → centre. **Skip-gram:** centre → context.

Word2vec is a family of shallow prediction models for learning word embeddings. CBOW pools surrounding input embeddings, then predicts the centre word. Skip-gram creates separate centre–context predictions. Both use shared lookup parameters across token occurrences and have a linear projection/lookup, not a nonlinear hidden activation. In the simple CBOW variant shown here, the context projection is the mean. The averaging operation is related to sentence mean pooling, but the training task and the represented span are different. Source: [Mikolov et al. (2013), Efficient Estimation](https://arxiv.org/abs/1301.3781).

## Inside skip-gram: two trainable tables

<img src="../img/word2vec_2027/architecture.svg" alt="One-hot centre x selects input row E_apples. Output table O scores all vocabulary contexts by dot products. Softmax gives a context distribution." width="1100">

$h=E^\top x=e_w$, $\quad z_c=o_c^\top e_w$. The lookup is the hidden representation; there is **no nonlinear hidden layer**.

E and O each have shape |V| by d. The same vocabulary appears in both but the parameters differ. x is a conceptual one-hot vector; an implementation indexes the row directly. O h is a length-|V| vector. The full softmax architecture makes the prediction problem explicit; the following slides replace its expensive objective with negative sampling. No bias is included in this standard exposition. CBOW changes h to the average of several E rows. The original implementation also supports hierarchical softmax, which is outside this lecture's scope.

## Where do the vectors come from at step 0?

<img src="../img/word2vec_2027/initialization.svg" alt="Input table E starts with small random values. Output table O starts with zeros in the original word2vec C implementation. All dot products are initially zero." width="1100">

**Original word2vec implementation:** random input vectors, zero output vectors. No semantic structure is supplied at initialization.

The original C InitNet initializes syn0 (E) to pseudorandom values approximately uniform on [-0.5/d, 0.5/d), and syn1neg (O for negative sampling) to zero. This is an implementation choice, not a requirement that all word2vec implementations initialize identically. The figure shows the first three coordinates of the eight-dimensional teaching run. At this initialization every score is zero, so full softmax predicts 1/|V| for every word and sigmoid predicts 1/2 for each pair. Word-frequency counts are still needed for the vocabulary, subsampling and noise distribution; they do not provide pretrained embedding coordinates. Source: [word2vec.c, InitNet](https://github.com/tmikolov/word2vec/blob/master/word2vec.c).

## Quiz: which table moves on the first update?

<img src="../img/word2vec_2027/init_quiz.svg" alt="The input vector e is nonzero and the output vectors are zero. The score is their dot product. Options: A input E only, B output O only, C both tables, D neither table." width="1100">

A prediction makes a nonzero error. Take one **stochastic gradient descent (SGD)** step. **Vote A–D.**

**B — only the output table $O$.** The input gradient contains the old output vectors, which are zero. Output gradients contain the nonzero input vector.

Later updates can change both tables. If **both** tables started at zero, this dot-product model would stay stuck.

Allow 30 seconds. Hint: differentiate o^T e with respect to each vector. For logistic loss, grad_e=(sigmoid(o^T e)-y)o and grad_o=(sigmoid(o^T e)-y)e. With all output rows zero, every contribution to grad_e is zero. With e nonzero, an observed positive pair gives a nonzero grad_o. A reverses these roles. C describes typical later steps but not the first one. D would hold if both sides were zero. A full-softmax step also changes only O under this initialization. Focus on gradients computed from the old parameters, not sequential in-place mutations.

## Full softmax: predict an observed context

<img src="../img/word2vec_2027/softmax.svg" alt="For centre apples, every context starts at probability one seventh. The observed context tasty is highlighted. Training raises its logit relative to the others." width="1100">

For the pair *(apples, tasty)*: $p(c\mid w)=\frac{\exp(o_c^\top e_w)}{\sum_{j\in V}\exp(o_j^\top e_w)}$, $\quad L=-\log p(\mathrm{tasty}\mid\mathrm{apples})$.

At initialization p=1/7 and loss=log(7)=1.946. The derivative with respect to logit c is p(c|w)-1[c=tasty]. It is -6/7 for tasty and +1/7 for each other word. This step pushes tasty up relative to the alternatives; other real contexts get their own positive training examples. Full softmax computes scores and gradients for every output row, costing O(|V|d) per pair. Repeated training shares each vector across many pairs.

## Skip-gram with negative sampling (SGNS)

<img src="../img/word2vec_2027/negative_sampling.svg" alt="Centre apples is paired with observed context tasty and sampled contexts furry and rabbits. Each pair has its own binary logistic loss, with labels one, zero and zero." width="1100">

$L=-\log\sigma(e_w^\top o_c)-\sum_{i=1}^{k}\log\sigma(-e_w^\top o_{n_i})$, where $\sigma(s)=1/(1+e^{-s})$.

Negative sampling replaces the normalized full-softmax objective with binary discrimination between observed pairs and noise pairs. Here k=2, centre apples is fixed, tasty is the positive context, and furry and rabbits are illustrative noise samples. The sigmoid scores are not a normalized context-word distribution. Noise is drawn from q(c) proportional to token frequency(c)^(3/4) in the original word2vec recipe. A sampled pair is not known to be semantically false or impossible: it can also occur elsewhere in the corpus. Some implementations reject a sample equal to the current positive context. The teaching code uses independent draws with replacement, including possible collisions, so that its expected objective is particularly simple. The cost is O((k+1)d) per observed pair rather than O(|V|d). Source: [Mikolov et al. (2013), Distributed Representations](https://arxiv.org/abs/1310.4546).

## Quiz: trace one update through the tables

<img src="../img/word2vec_2027/rows_quiz.svg" alt="SGNS pair: apples to tasty, with negative contexts furry and rabbits. Four options list which rows of E and O receive gradients." width="1100">

We are **past initialization**. Which rows receive gradients from this one training example?

**C — input row E[apples]; output rows O[tasty], O[furry], O[rabbits].**

The other input and output rows receive no gradient from this example. A word's input row and output row are separate parameters.

Give 30–45 seconds for pairs to trace the arrows. Assume generic nonzero weights, no regularization, and plain stochastic gradient descent. A (all rows of both tables) confuses parameter sharing over the whole corpus with one example. B (the four words in E only) forgets O and mixes the two roles. C is correct. D (E[apples] plus every row of O) describes full softmax, not this negative-sampling example. Hint: list the vector lookups needed to calculate the three dot products. We say receive gradients to describe the dependency; exact cancellations can make an individual numerical gradient zero.

## A worked gradient step: make the scores improve

<img src="../img/word2vec_2027/gradient_step.svg" alt="A two-dimensional example after initialization. Input apples moves from (1,0) to (1,0.2). Positive tasty output moves from (0,1) to (0.1,1), and noise furry output moves from (0,-1) to (-0.1,-1). Scores become 0.3 and -0.3; loss drops from 1.386 to 1.109." width="1100">

**Stochastic gradient descent (SGD):** $\theta\leftarrow\theta-\eta\nabla_\theta L$. Use the old vectors to compute every gradient.

For each pair, $g=\sigma(o^\top e)-y$: input gradient $go$; output gradient $ge$. Add the input gradients over the positive and noise pairs.

This separate hand-worked example has d=2, k=1 and learning rate eta=0.2; it is one step after initialization, not the first step of the eight-dimensional training run. Start with e=(1,0), o+=(0,1), o-=(0,-1). Both logits are zero, so sigmoid scores are 0.5 and errors are g+=-0.5 and g-=+0.5. The input gradient is g+ o+ + g- o- = (0,-1). Output gradients are g+ e=(-0.5,0) and g- e=(0.5,0). Simultaneously updating from the old values gives e'=(1,0.2), o+'=(0.1,1), o-'=(-0.1,-1). New dot products are +0.3 and -0.3, new sigmoid scores are 0.57444 and 0.42556, and the loss falls from 2 log(2)=1.38629 to 2 log(1+exp(-0.3))=1.10871. This directly changes the input–output dot products; it does not directly pull the two input rows E[apples] and E[tasty] together.

## Training repeats this small operation

<img src="../img/word2vec_2027/training_loop.svg" alt="Build a vocabulary and word-frequency distribution, initialize E and O once, extract window pairs, sample noise, compute loss and gradients, update selected rows, repeat over the corpus, then retain word vectors." width="1100">

Different examples share the same rows. Repeated prediction errors shape the embedding space.

An epoch is one pass over the observed pairs. The original implementation streams windows from a large corpus, may subsample frequent tokens, varies the effective window radius, and decays the learning rate. Our tiny example fixes the window and omits subsampling so that the relationship with its count matrix is exact and inspectable. It uses 8 dimensions, 2 negative draws per positive, 400 epochs, and NumPy random seed 7. We generate window pairs from the corpus directly; constructing a full count matrix is not a prerequisite for word2vec training. The next slide's loss is the exact expected negative-sampling objective under the toy noise distribution, not the loss of just the last sampled example.

In [ ]:
# Reproducible toy SGNS experiment; rebuild the figures with:
# python scripts/build_word2vec_figures.py  (from the repository root)
import numpy as np

vocabulary = ["apples", "oranges", "rabbits", "hamsters", "are", "tasty", "furry"]
word_id = {word: i for i, word in enumerate(vocabulary)}
corpus = (["apples are tasty"] * 4 + ["oranges are tasty"] * 4
          + ["rabbits are furry"] * 2 + ["hamsters are furry"] * 2)
window_radius = 2
pairs = []
token_counts = np.zeros(len(vocabulary), dtype=int)
for sentence in corpus:
    ids = [word_id[word] for word in sentence.split()]
    np.add.at(token_counts, ids, 1)
    for position, centre in enumerate(ids):
        for context_position in range(max(0, position - window_radius),
                                      min(len(ids), position + window_radius + 1)):
            if position != context_position:
                pairs.append((centre, ids[context_position]))
# Sorting only fixes a reproducible starting order before epoch-wise shuffling.
pairs = np.array(sorted(pairs), dtype=int)
# C is for analysis and figures, not a prerequisite for the training updates.
C = np.zeros((len(vocabulary), len(vocabulary)), dtype=int)
np.add.at(C, (pairs[:, 0], pairs[:, 1]), 1)

rng = np.random.default_rng(7)
dimensions, negative_samples, epochs = 8, 2, 400
E = rng.uniform(-0.5 / dimensions, 0.5 / dimensions,
                (len(vocabulary), dimensions))
O = np.zeros_like(E)
initial_E = E.copy()
noise_distribution = token_counts.astype(float) ** 0.75
noise_distribution /= noise_distribution.sum()

def expected_sgns_loss(input_table, output_table):
    scores = input_table @ output_table.T
    positive = C * np.logaddexp(0, -scores)
    noise_weights = negative_samples * C.sum(axis=1)[:, None] * noise_distribution
    negative = noise_weights * np.logaddexp(0, scores)
    return (positive + negative).sum() / len(pairs)

loss_history = [expected_sgns_loss(E, O)]
for epoch in range(epochs):
    learning_rate = 0.06 * (1 - epoch / epochs) + 0.005
    for pair_index in rng.permutation(len(pairs)):
        centre, context = pairs[pair_index]
        # Independent noise draws with replacement, including possible collisions.
        output_rows = np.r_[context, rng.choice(len(vocabulary), negative_samples,
                                                p=noise_distribution)]
        labels = np.r_[1.0, np.zeros(negative_samples)]
        old_input = E[centre].copy()
        old_outputs = O[output_rows].copy()
        scores = old_outputs @ old_input
        errors = 1 / (1 + np.exp(-scores)) - labels
        E[centre] -= learning_rate * (errors @ old_outputs)
        # add.at correctly accumulates updates when a sampled row is repeated.
        np.add.at(O, output_rows, -learning_rate * errors[:, None] * old_input)
    loss_history.append(expected_sgns_loss(E, O))

def cosine_matrix(vectors):
    unit_vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
    return unit_vectors @ unit_vectors.T

initial_cosines = cosine_matrix(initial_E[:4])
trained_cosines = cosine_matrix(E[:4])

## From random rows to shared-context similarity

<img src="../img/word2vec_2027/training_results.svg" alt="Actual toy training: noun-pair cosine heatmaps before and after training, with the exact expected negative-sampling loss across 400 epochs. Apples and oranges become similar, as do rabbits and hamsters." width="1100">

Eight-dimensional input vectors from our 12-sentence corpus. These are **measured toy results**, not a general benchmark.

The toy run uses eight dimensions for seven vocabulary items to illustrate learning; it is not a compression example. Real word2vec vocabularies are usually much larger than the embedding dimension. The two pairs with identical context distributions end up with high input–input cosine similarity even though the words in each pair never directly co-occur. Read the measured values in the heatmaps; similarity is not guaranteed to match a hand-labelled ontology or every random run. The loss curve can fluctuate because each update uses sampled negatives. More dimensions and more epochs are not automatically better on real data. Here the vocabulary and corpus are deliberately tiny enough to inspect. All values and figures are regenerated by scripts/build_word2vec_figures.py from the tagged code cell above.

## After training, what is the word embedding?

<img src="../img/word2vec_2027/export_vectors.svg" alt="Training uses dot products between rows of E and O. A common downstream choice keeps E, compares its rows by cosine, and averages them for sentence representations." width="1100">

**Common choice:** keep the input table $E$. Training uses input–output scores; word similarity usually compares input–input vectors.

The original word2vec tool writes syn0 (the input table) as word vectors. Other implementations or applications may use output vectors or combine the two. Specify the convention. There is no extra hidden embedding matrix behind E and O. At inference a static word vector is a lookup, not a new prediction or a forward pass over the whole corpus.

## Count-based and prediction-based: shared evidence

<img src="../img/word2vec_2027/shared_evidence.svg" alt="The same context pairs can be accumulated into a matrix and factorized, or sampled to train a prediction objective. Both routes can yield dense vectors. There is no count-matrix initialization arrow into word2vec." width="1100">

Both use **distributional evidence**. They differ in the objective, weighting and optimization—not in whether the corpus contains counts.

This is neither a claim that count-based and prediction-based methods are unrelated nor that they are identical. Context definition and preprocessing strongly influence both. Count-based models explicitly summarize statistics before their representation objective; prediction-based models can learn from individual examples. Dense versus sparse is a separate distinction. GloVe is a useful bridge because it learns dense parameters by fitting log co-occurrence counts. Sources: [Pennington et al. (2014)](https://aclanthology.org/D14-1162/) and [Levy et al. (2015)](https://aclanthology.org/Q15-1016/).

## The mathematical bridge: SGNS fits a score matrix

<img src="../img/word2vec_2027/implicit_matrix.svg" alt="The product of the input and output embedding tables is a low-rank word-context score matrix. With unigram noise and independently optimized scores the target is PMI minus log k." width="1100">

With noise $q(c)=P_D(c)$, the independent-score optimum is $e_w^\top o_c=\operatorname{PMI}(w,c)-\log k$ **for observed pairs**.

This is an **implicit, weighted matrix-factorization connection**. A small embedding dimension, different noise distribution and different objective weights change the fit; SGNS is not the same algorithm as SVD.

Pointwise mutual information was introduced in the count-based section. Let C_wc be an observed pair count, C_w the centre count over pairs, q(c) the noise distribution and k the number of negatives per positive. The expected contribution of one score s is C_wc log sigma(s) + k C_w q(c) log sigma(-s). If each score can vary independently, differentiating yields s*=log[C_wc/(k C_w q(c))] = log[P_D(c|w)/(k q(c))]. Taking q(c)=P_D(c), the empirical context marginal, gives PMI(w,c)-log k. For zero-count pairs the unconstrained optimum tends to negative infinity. A finite-dimensional product E O^T couples scores and approximates them under the SGNS weighting; it does not independently attain every optimum. The usual q proportional to frequency^0.75 changes the shift to depend on c. Input and output factorizations need not give identical input cosine geometry even when score matrices agree. Source: [Levy & Goldberg (2014), Neural Word Embedding as Implicit Matrix Factorization](https://papers.nips.cc/paper_files/paper/2014/file/b78666971ceae55a8e87efb7cbfd9ad4-Paper.pdf).

## Different routes to a word representation

| Method | What it fits or stores | Word vectors |
| :--- | :--- | :--- |
| Raw context counts | Observed pair counts | Usually sparse rows |
| PPMI + truncated SVD | A low-rank approximation to weighted counts | Dense |
| **Global Vectors (GloVe)** | Weighted squared error on log counts, with bias terms | Dense |
| SGNS | Observed pairs versus sampled noise | Dense |

**Dense does not imply prediction-based. Shared evidence does not imply identical vectors.**

GloVe minimizes sum over observed pairs f(C_wc)(e_w^T o_c+b_w+b_c-log C_wc)^2, with a weighting function that limits the influence of very frequent co-occurrences. PPMI+SVD instead minimizes a low-rank reconstruction error for its transformed matrix. SGNS weights the positive and negative logistic terms according to pair and noise frequencies. Choices of window, weighting, dimension and evaluation matter; no method wins merely because it is called neural. Source: [GloVe paper](https://aclanthology.org/D14-1162/) and [Levy et al. (2015)](https://aclanthology.org/Q15-1016/).

## Learned word embeddings in two dimensions

<img src="../img/word_representations.svg" alt="Two-dimensional projection of word embeddings, showing clusters of related words" style="height:360px;width:auto;">

A two-dimensional **t-distributed stochastic neighbor embedding (t-SNE)** projection can distort distances. Test similarity claims in the **original embedding space**.

## Static word embeddings have limits

The word *bank* receives the same vector in “river bank” and “bank loan”. Word order is also absent from a single lookup vector.

The next two lectures introduce models that build a new representation from the surrounding text.


## Mean pooling of static word embeddings

For a sentence $s=(w_1,\ldots,w_n)$, average the static embedding of every word token:

$$f(s)=\frac{1}{n}\sum_{i=1}^{n} f(w_i)$$

The result is a **sentence embedding**: one fixed-size vector for the whole sentence. It can be used for similarity or as input to a classifier.


## CBOW training and sentence mean pooling

<img src="../img/word2vec_2027/pooling_comparison.svg" alt="CBOW averages surrounding word vectors then predicts a centre word and updates embeddings. Sentence mean pooling averages the word vectors of the whole sentence to produce a reusable sentence representation." width="1100">

The averaging operation is similar; the **learning task and the span being represented** are different.

## What mean pooling loses

Toy static word vectors illustrate the loss of word order. Who chases whom changes, but the sentence embedding stays the same.


In [5]:
import numpy as np

word_vectors = {
    "dogs": [1., 0.], "chase": [0., 1.], "cats": [.8, .2]
}

def mean_pool(sentence):
    vectors = [word_vectors[w] for w in sentence.split()]
    return np.mean(vectors, axis=0)

for sentence in ("dogs chase cats", "cats chase dogs"):
    print(sentence, mean_pool(sentence))

dogs chase cats [0.6 0.4]
cats chase dogs [0.6 0.4]


## Mean pooling in the library

The **Sentence Transformers library** can average static GloVe word embeddings:

```python
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/average_word_embeddings_glove.6B.300d"
)
sentences = ["dogs chase cats", "cats chase dogs"]
vectors = model.encode(sentences)
```

Each sentence receives a 300-dimensional mean of word vectors.

The [`average_word_embeddings_glove.6B.300d` model](https://huggingface.co/sentence-transformers/average_word_embeddings_glove.6B.300d) contains a word-embedding lookup and a mean-pooling layer. `SentenceTransformer` is the library interface used to load it. The model has no Transformer layers.


## Summary

- Word2vec starts with uninformative parameters; context-prediction errors train two embedding tables.
- CBOW predicts a centre word from pooled context; skip-gram predicts contexts from the centre.
- Count-based and prediction-based methods share evidence and have mathematical connections, but different objectives.
- Cosine measures direction. Mean pooling gives a sentence vector, losing word order and contextual word senses.

## Additional reading

- Mikolov et al. (2013): [word2vec architectures](https://arxiv.org/abs/1301.3781) and [negative sampling](https://arxiv.org/abs/1310.4546)
- [Original word2vec implementation](https://github.com/tmikolov/word2vec/blob/master/word2vec.c): see `InitNet` and `TrainModelThread`
- Levy & Goldberg (2014): [SGNS as implicit matrix factorization](https://papers.nips.cc/paper_files/paper/2014/file/b78666971ceae55a8e87efb7cbfd9ad4-Paper.pdf)
- Pennington et al. (2014): [GloVe](https://aclanthology.org/D14-1162/)
- Levy et al. (2015): [Comparing distributional methods](https://aclanthology.org/Q15-1016/)
- [Pretrained GloVe mean-pooling model](https://huggingface.co/sentence-transformers/average_word_embeddings_glove.6B.300d)